In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [2]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [3]:
# import the datasets
df = pd.read_csv(config["output_data"]["file6"])

In [4]:
# drop unamed
df = df.drop(columns=['Unnamed: 0'])
df.shape

(154358, 16)

## Ordering values

In [63]:
# sort values
df = df.sort_values(by=['client_id', 'visit_id', 'date_time'], ascending=True)
df.head(5)

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,first_start_time,last_confirm_time
87485,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34
87486,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34
87487,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34
87488,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34
87489,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34


## Mapping steps and calculate difference

In [64]:
# mapping step to integer to calculate difference between steps
step_mapping = {
    'start': 0,
    'step_1': 1,
    'step_2': 2,
    'step_3': 3,
    'confirm': 4
}

# create a new column step to map step 1 = 1...
df['step'] = df['process_step'].map(step_mapping)

# create a next step column and retrieve 1 step 
df['next_step'] = df.groupby(['visit_id'])['step'].shift(-1)

# calculate the difference
df['step_diff'] = df['next_step']- df['step']

df

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,first_start_time,last_confirm_time,step,next_step,step_diff
87485,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34,0,1.0,1.0
87486,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34,1,2.0,1.0
87487,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34,2,3.0,1.0
87488,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34,3,4.0,1.0
87489,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,2017-04-15 12:57:56,2017-04-15 13:00:34,4,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31084,9999400,915967319_19082377501,288904166_90826265353_276104,start,2017-04-20 05:21:28,7.0,86.0,28.5,U,2.0,51787.04,0.0,3.0,Test,2017-04-20 05:21:28,2017-04-20 05:23:27,0,1.0,1.0
31085,9999400,915967319_19082377501,288904166_90826265353_276104,step_1,2017-04-20 05:21:50,7.0,86.0,28.5,U,2.0,51787.04,0.0,3.0,Test,2017-04-20 05:21:28,2017-04-20 05:23:27,1,2.0,1.0
31086,9999400,915967319_19082377501,288904166_90826265353_276104,step_2,2017-04-20 05:22:17,7.0,86.0,28.5,U,2.0,51787.04,0.0,3.0,Test,2017-04-20 05:21:28,2017-04-20 05:23:27,2,3.0,1.0
31087,9999400,915967319_19082377501,288904166_90826265353_276104,step_3,2017-04-20 05:23:03,7.0,86.0,28.5,U,2.0,51787.04,0.0,3.0,Test,2017-04-20 05:21:28,2017-04-20 05:23:27,3,4.0,1.0


## CONTROL COMPLETION RATE

In [100]:
# remove confirm (at there is no other step after so it does not need to be achieved)

df_control = df[df['Variation'] == 'Control']

kp1_by_step_control = df_control[df_control['process_step'] !='confirm']

# group by process_step
kp1_by_step_control = kp1_by_step_control.groupby('process_step').agg(
    total=('process_step', 'size'),           # total number of rows of step1 for ex
    count_step=('step_diff', lambda x: (x == 1).sum())  # number of times step_diff == 1 per step
)

# calculate completion rate per step
kp1_by_step_control['pct_step_control'] = round(kp1_by_step_control['count_step'] / kp1_by_step_control['total']*100,0)

kp1_by_step_control


,total,count_step,pct_step_control
process_step,,,
start,11678,11624,100.0
step_1,12807,12164,95.0
step_2,14194,13222,93.0
step_3,13789,11015,80.0


In [101]:
# list of the correct sequence
funnel = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

# function to check if the sequence exsits in a visit
def has_complete_funnel(visit_df):
    # sort the visit chronologically
    steps = visit_df.sort_values('date_time')['process_step'].tolist()
    
    funnel_index = 0  # pointer to track position in funnel
    for step in steps:
        if step == funnel[funnel_index]:
            funnel_index += 1
            if funnel_index == len(funnel):  # all steps found
                return True
    return False  # funnel not complete

# per visit_id
funnel_per_visit_control = df_control.groupby('visit_id').apply(has_complete_funnel).reset_index()
funnel_per_visit_control.columns = ['visit_id', 'completed_process']


funnel_per_visit_control

,visit_id,completed_process
0,10006594_66157970412_679648,True
1,100254180_47139859079_984581,True
2,100309269_21684743336_936307,True
3,100481857_71511233596_788753,True
4,100733473_61604582110_215085,True
...,...,...
11016,999512351_9425260762_890767,True
11017,999528108_94761236019_731649,True
11018,999528902_49133507319_516085,True
11019,999859408_41720215615_938916,True


In [116]:
# calculate number of visits that have at least one completed sequence (from start to confirm without errors)
num_true = funnel_per_visit_control['completed_process'].value_counts()[True]
control_completion_rate = round((num_true / len(funnel_per_visit_control['completed_process']))*100,2)
control_completion_rate

np.float64(99.14)

## TEST COMPLETION RATE

In [107]:
# remove confirm (at there is no other step after so it does not need to be achieved)

df_test = df[df['Variation'] == 'Test']

kp1_by_step_test = df_test[df_test['process_step'] !='confirm']

# group by process_step
kp1_by_step_test = kp1_by_step_test.groupby('process_step').agg(
    total=('process_step', 'size'),           # total number of rows of step1 for ex
    count_step=('step_diff', lambda x: (x == 1).sum())  # number of times step_diff == 1 per step
)

# calculate completion rate per step
kp1_by_step_test['pct_step_test'] = round(kp1_by_step_test['count_step'] / kp1_by_step_test['total']*100,0)

kp1_by_step_test

,total,count_step,pct_step_test
process_step,,,
start,17053,16945,99.0
step_1,18669,15979,86.0
step_2,17082,15674,92.0
step_3,16078,14353,89.0


In [105]:
# list of the correct sequence
funnel = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

# function to check if the sequence exsits in a visit
def has_complete_funnel(visit_df):
    # sort the visit chronologically
    steps = visit_df.sort_values('date_time')['process_step'].tolist()
    
    funnel_index = 0  # pointer to track position in funnel
    for step in steps:
        if step == funnel[funnel_index]:
            funnel_index += 1
            if funnel_index == len(funnel):  # all steps found
                return True
    return False  # funnel not complete

# per visit_id
funnel_per_visit_test = df_test.groupby('visit_id').apply(has_complete_funnel).reset_index()
funnel_per_visit_test.columns = ['visit_id', 'completed_process']


funnel_per_visit_test

,visit_id,completed_process
0,100019538_17884295066_43909,True
1,100173292_91322748906_143563,True
2,100217156_67053490690_383412,True
3,100258507_71262593004_214494,True
4,100293270_70949014562_696814,True
...,...,...
14345,999954858_74676709104_879685,True
14346,999958344_67534252886_39917,True
14347,999971096_28827267783_236076,True
14348,999976049_95772503197_182554,True


In [115]:
# calculate number of visits that have at least one completed sequence (from start to confirm without errors)
num_true = funnel_per_visit_test['completed_process'].value_counts()[True]
test_completion_rate = round((num_true / len(funnel_per_visit_test['completed_process']))*100,2)
test_completion_rate

np.float64(99.73)

In [111]:
# merge Control et Test by process_step
KP1_steps = kp1_by_step_control.merge(
    kp1_by_step_test,
    on='process_step',
    suffixes=('_Control', '_Test')
)
KP1_steps

,total_Control,count_step_Control,pct_step_control,total_Test,count_step_Test,pct_step_test
process_step,,,,,,
start,11678,11624,100.0,17053,16945,99.0
step_1,12807,12164,95.0,18669,15979,86.0
step_2,14194,13222,93.0,17082,15674,92.0
step_3,13789,11015,80.0,16078,14353,89.0


In [2]:
print(f" Completion rate for Control Group is: {control_completion_rate} %")
print(f" Completion rate for Test Group is: {control_test_rate} %")

NameError: name 'control_completion_rate' is not defined